# Joint optimization of macro-F1 decision weights
Uses the existing ten Pipeline/ColumnTransformer models. Fits no new classifier. Selects weights on fold 0 only and freezes before confirmation. Preserves the 0.59242 leaderboard submission.


In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import joblib
from scipy.optimize import differential_evolution
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
root=Path.cwd() if (Path.cwd()/'scripts').is_dir() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
from scripts.train import macro_f1
from scripts.predict import FoldPipelineEnsemble

saved=np.load(root/'reports/improved_predictions.npz')
y=saved['y']; probability=saved['oof']; old_weights=saved['decision_weights']
splits=list(StratifiedKFold(5,shuffle=True,random_state=42).split(np.zeros(len(y)),y))
selection=splits[0][1]
confirmation=np.concatenate([v for _,v in splits[1:]])
selection_probability=probability[selection]
selection_y=y[selection]

def objective(log_weights):
    """Minimize negative macro-F1 on selection fold only, with Grade 5 as anchor."""
    weights=np.r_[np.exp(log_weights),1.0]
    return -macro_f1(selection_y,(selection_probability*weights).argmax(1)+1)

trials=[]
for seed in [42,137,2026]:
    result=differential_evolution(objective,[(np.log(.4),np.log(1.8))]*4,
        seed=seed,popsize=12,maxiter=70,tol=1e-7,polish=False,
        x0=np.log(old_weights[:4]),workers=1)
    weights=np.r_[np.exp(result.x),1.0]
    trials.append({'seed':seed,'selection_macro_f1':-result.fun,'weights':weights.tolist()})
    print(json.dumps(trials[-1]),flush=True)
old_selection=macro_f1(selection_y,(selection_probability*old_weights).argmax(1)+1)
trials.append({'seed':'incumbent','selection_macro_f1':old_selection,'weights':old_weights.tolist()})
best=max(trials,key=lambda row:row['selection_macro_f1'])
weights=np.array(best['weights'])
# Freeze before measuring confirmation folds.
(root/'reports/calibrated_frozen_selection.json').write_text(json.dumps({'trials':trials,'best':best},indent=2))
pred=(probability*weights).argmax(1)+1
old_pred=(probability*old_weights).argmax(1)+1
report={'cv_macro_f1':f1_score(y,pred,average='macro'),
    'confirmation_macro_f1':f1_score(y[confirmation],pred[confirmation],average='macro'),
    'baseline_cv_macro_f1':f1_score(y,old_pred,average='macro'),
    'baseline_confirmation_macro_f1':f1_score(y[confirmation],old_pred[confirmation],average='macro'),
    'selection':best,'classification_report':classification_report(y,pred,output_dict=True),
    'protocol':'Three deterministic joint searches for four relative decision multipliers on fold 0 only; frozen for folds 1-4. Same pretrained sklearn Pipeline ensemble. Reused folds make this internal validation, not an unbiased new benchmark.',
    'baseline_leaderboard_user_reported':0.59242,'new_leaderboard':None}
report['confirmation_improved']=report['confirmation_macro_f1']>report['baseline_confirmation_macro_f1']
(root/'reports/calibrated_model_metrics.json').write_text(json.dumps(report,indent=2))
rows=[]
for fold,(_,v) in enumerate(splits):
    rows.append({'fold':fold,'role':'selection' if fold==0 else 'confirmation',
        'baseline_macro_f1':f1_score(y[v],old_pred[v],average='macro'),
        'macro_f1':f1_score(y[v],pred[v],average='macro')})
pd.DataFrame(rows).to_csv(root/'reports/calibrated_fold_scores.csv',index=False)
if report['confirmation_improved']:
    incumbent=joblib.load(root/'models/improved_ensemble.pkl')
    model=FoldPipelineEnsemble(incumbent.model_files,weights.tolist())
    joblib.dump(model,root/'models/calibrated_ensemble.pkl')
    test=pd.read_csv(root/'data/test.csv');sample=pd.read_csv(root/'data/sample_submission.csv')
    assert sample.building_id.is_unique and test.building_id.is_unique
    assert set(sample.building_id)==set(test.building_id)
    mapping=pd.Series((saved['test']*weights).argmax(1)+1,index=test.building_id)
    sample['damage_grade']=sample.building_id.map(mapping)
    assert not sample.isna().any().any() and sample.damage_grade.isin(range(1,6)).all()
    sample.damage_grade=sample.damage_grade.astype('int64')
    sample.to_csv(root/'submissions/submission_calibrated.csv',index=False)
print(json.dumps(report,indent=2),flush=True)
